In [0]:
print("")

In [0]:
from pathlib import Path
import numpy as np  
import pandas as pd

#Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")

In [0]:
pickle_path = DATA_DIR / "all_pages.pkl"

In [0]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import pickle
import time

search_bases = [
    "https://www.rottentomatoes.com/browse/movies_at_home/affiliates:fandango-at-home",
    "https://www.rottentomatoes.com/browse/movies_at_home/affiliates:netflix",
    "https://www.rottentomatoes.com/browse/movies_at_home/affiliates:apple-tv-plus",
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_series_urls = []
count_error = 0

for base_url in search_bases:
    for page in range(1, 12):  # páginas 1 a 11
        url = f"{base_url}?page={page}"
        # print(f"Scrapeando página {page}: {url}")

        resp = requests.get(url, headers=headers, timeout=10)

        
        if resp.status_code != 200:
            count_error += 1
            ##### Error if the page has not been created
            print(f"Error en página {page} de {base_url}: HTTP {resp.status_code}") 
            #print(count_error)
            continue # If the error happens, continue with next page

        soup = BeautifulSoup(resp.content, "html.parser")

        for a in soup.find_all("a", href=True):
            full_url = urljoin(url, a["href"])

            if "rottentomatoes.com/tv/" in full_url:
                all_series_urls.append(full_url)
            elif"rottentomatoes.com/m/" in full_url:
                all_series_urls.append(full_url)

        time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_series_urls = list(dict.fromkeys(all_series_urls))


print(f"\nTotal de series encontradas: {len(all_series_urls)}")
for u in all_series_urls:
    print(u)

with open(DATA_DIR / "all_pages.pkl", "wb") as f:
    pickle.dump(all_series_urls, f)


# Function to take many URLs

In [0]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import pickle
import time

url_search = [
    "https://www.rottentomatoes.com/browse/tv_series_browse/?page="
]

def generate_search_page_urls(base_urls, start=1, end=4):
    """Genera una lista de URLs de páginas de búsqueda a partir de una lista base."""
    return [f"{base}{page}" for base in base_urls for page in range(start, end + 1)]


def collect_series_urls_from_search_pages(page_urls, delay=1):
    """Extrae todas las URLs de series de una lista de páginas de búsqueda."""
    all_series_urls = []

    for page_url in page_urls:
        resp = requests.get(page_url, headers=headers, timeout=10)
        if resp.status_code != 200:
            print(f"Error en página {page_url}: HTTP {resp.status_code}")
            continue

        soup = BeautifulSoup(resp.content, "html.parser")
        for a in soup.find_all("a", href=True):
            full_url = urljoin(page_url, a["href"])
            if "/tv/" in full_url:
                all_series_urls.append(full_url)
            elif "rottentomatoes.com/tv/" in full_url:
                all_series_urls.append(full_url)

        time.sleep(delay)

    return list(dict.fromkeys(all_series_urls))


# Generar las URLs de todas las páginas de búsqueda y luego extraer las URLs de series.
search_page_urls = generate_search_page_urls(url_search, start=1, end=4)
print(f"Se generaron {len(search_page_urls)} URLs de búsqueda")

all_series_urls = collect_series_urls_from_search_pages(search_page_urls, delay=1)
print(f"\nTotal de series encontradas: {len(all_series_urls)}")
for u in all_series_urls:
    print(u)

with open(DATA_DIR / "series_urls_pages.pkl", "wb") as f:
    pickle.dump(all_series_urls, f)


In [0]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import pickle
import time



url_search = [
    "https://www.rottentomatoes.com/browse/tv_series_browse/?page="
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_series_urls = []

# for url in range(0, len(url_search)):
for page in range(1, 5):  # 1 hasta 11
    url = f"https://www.rottentomatoes.com/browse/tv_series_browse/?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/tv/" in full_url:
            all_series_urls.append(full_url)

    time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_series_urls = list(dict.fromkeys(all_series_urls))

print(f"\nTotal de series encontradas: {len(all_series_urls)}")

for u in all_series_urls:
    print(u)


with open("series_urls_pages.pkl", "wb") as f:
    pickle.dump(all_series_urls, f)

# with open("series_urls_pages.json", "w", encoding="utf-8") as f:
#     json.dump(all_series_urls, f, indent=4, ensure_ascii=False)
###### Si ya esta guardado como JSON
# with open("series_urls_pages.json", "r", encoding="utf-8") as f:
#     all_series_urls = json.load(f)

# with open("series_urls_pages.pkl", "wb") as f:
    # pickle.dump(all_series_urls, f)

In [0]:
# Path for files pkl wich are smaller than JSON files
# This File will contain the dataset of Series
pickle_path = DATA_DIR / "series_urls.pkl"

# Function to take all the values from many URLS

In [0]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd
import time


# "User-Agent" REQUEST simula que la petición como si viniera desde un navegador Chrome en Windows para evitar bloqueos de HTML
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}



###########################    DEFINICIÓN DE MI FUNCIÓN PARA CADA URL    ###########################
def scrape_movie(url):
    resp = requests.get(url, headers=headers, timeout=10) ## Uso de el agente para hacer la solicitud como si viniera del navegador

    if resp.status_code != 200:
        print(f"Error HTTP {resp.status_code}: {url}")
        return None

    soup = BeautifulSoup(resp.content, "html.parser")

    # Título 
    title = None
    h1 = soup.find("h1")

    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]

####################### 0. DETERMINE CONTENT TYPE (Movie or TV Show)

    if "rottentomatoes.com/m/" in url:
        content_type = "Movie"
    elif "rottentomatoes.com/tv/" in url:
        content_type = "TV Show"
    else:
        content_type = "Unknown"

####################### 1 . CALLING GENRE(S) 

    genre = None # create the variable

    script = soup.find("script", id="mps-page-integration")

    if script:
        script_text = script.get_text()

        match = re.search(r'"cag\[genre\]":"([^"]+)"', script_text) #Looking for the text

        if match:
            genre = match.group(1)
    
    genres = [] #Save genres

    if genre:
        genres = genre.split("|") # Spplit them 

    # print("Genre:", genre)
    
################# 2. CALLING: Summary, Tomatometer, Popcornmeter

     # Variables 
    summary = None
    tomatometer = None
    popcornmeter = None

    script = soup.find("script", id="media-scorecard-json")

    if script:
        try:
            data = json.loads(script.get_text(strip=True))
            
            #Checking for texts, different forms to get the values
            summary = data.get("description")
            tomatometer = data.get("criticsScore", {}).get("scorePercent")
            popcornmeter = data.get("audienceScore", {}).get("scorePercent")

        except json.JSONDecodeError:
            pass

 ############### 3. STREAM PLATFORMS 
    platform_names = []

    for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
        a = li.find("a", href=True)

        if a and "affiliates:" in a["href"]:
            name = a.get_text(strip=True)
            platform_names.append(name)

    platform_names = list(dict.fromkeys(platform_names))

############### 4. CRITICS CONSENSUS
    critics_consensus = None

    consensus_div = soup.find("div", id="critics-consensus")

    if consensus_div:
        p = consensus_div.find("p")
        if p:
            critics_consensus = p.get_text(" ", strip=True)

    movie_row = {
        "title": title,
        "content_type": content_type,
        "genre": genre,
        "url": url,
        "tomatometer": tomatometer,
        "popcornmeter": popcornmeter,
        "summary": summary,
        "critics_consensus": critics_consensus,
        "platforms": ", ".join(platform_names)
    }

    return movie_row ## Me retorna el valor de cada columna nueva


################################# Fin de la función ######################

In [0]:
#Cargo mi archivo pkl ### MAS ADELANTE SE ACTUALIZA CON TODAS LAS SERIES Y PEL[ICULAS]
with pickle_path.open("rb") as f:
    all_series_urls = pickle.load(f)

In [0]:
#####  For para crear la matriz
rows = []

for i, url in enumerate(all_series_urls, start=1):
    # print(f"Scrapeando {i}/{len(movie_urls)}: {url}")

    movie_row = scrape_movie(url)

    if movie_row is not None:
        rows.append(movie_row)

    time.sleep(1)


#### Creación del DataFrame 
df = pd.DataFrame(rows)



#### Guardar EL dataframe
df.to_csv("rottentomatoes_series_dataset.csv", index=False, encoding="utf-8")

df.head()